# Configuration

In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
1
0
NVIDIA L4


In [14]:
from pathlib import Path
import tensorflow as tf

# Data loading

In [15]:
from pathlib import Path
import tensorflow as tf

# --- Determine project root ---
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

# --- Paths to data ---
wikiart_path = project_root / "data"
train_dir = wikiart_path / "train"
val_dir = wikiart_path / "validation"
test_dir = wikiart_path / "test"

# --- Check paths exist ---
for path in [train_dir, val_dir, test_dir]:
    if not path.exists():
        print(f"Warning: {path} does not exist!")

print(f"Project root: {project_root}")
print(f"WikiArt path: {wikiart_path}")

# --- Dataset parameters ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)  # Resize images for model input

# --- Load datasets ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

print("Datasets loaded successfully!")
print(f"Train batches: {len(train_ds)}, Validation batches: {len(val_ds)}, Test batches: {len(test_ds)}")

Project root: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026
WikiArt path: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026/data
Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.
Datasets loaded successfully!
Train batches: 292, Validation batches: 63, Test batches: 64


In [19]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image

# --- Load processor ---
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

# --- Create a PyTorch Dataset from folders ---
class ImageFolderDataset(Dataset):
    def __init__(self, path, processor, transform=None):
        self.path = Path(path)
        self.processor = processor
        self.transform = transform
        self.samples = []
        self.classes = sorted([d.name for d in path.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        for cls in self.classes:
            cls_dir = path / cls
            for img_path in cls_dir.glob('*'):
                self.samples.append((img_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # ViT processor converts PIL image to tensor and normalizes
        encoding = self.processor(images=image, return_tensors="pt")
        pixel_values = encoding['pixel_values'].squeeze()  # remove batch dim
        return pixel_values, label

# --- Define transform (optional) ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ViT expects 224x224
])

# --- Create datasets ---
train_dataset = ImageFolderDataset(train_dir, processor, transform)
val_dataset = ImageFolderDataset(val_dir, processor, transform)

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

test_dataset = ImageFolderDataset(test_dir, processor, transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

# Version 1

In [20]:
from transformers import ViTForImageClassification

num_labels = len(train_dataset.classes)
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    id2label={i: cls for i, cls in enumerate(train_dataset.classes)},
    label2id={cls: i for i, cls in enumerate(train_dataset.classes)},
    ignore_mismatched_sizes=True  # THIS fixes the issue
)

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [20]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from sklearn.metrics import f1_score
import time
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = CrossEntropyLoss()
epochs = 10  # updated from 3 to 10

# Directory to save checkpoints
checkpoint_dir = "checkpoints/v1"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    start_time = time.time()
    
    # --- Training ---
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    for pixel_values, labels in train_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)

            val_loss += loss.item()
            val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # --- Epoch summary ---
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} - time: {epoch_time:.1f}s - "
          f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
          f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}")

    # --- Save checkpoint ---
    checkpoint_path = os.path.join(checkpoint_dir, f"vit_epoch{epoch+1}.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}\n")

Epoch 1/10 - time: 209.7s - loss: 1.1828 - f1: 0.6373 - val_loss: 0.6771 - val_f1: 0.7922
Checkpoint saved: checkpoints/vit_epoch1.pt

Epoch 2/10 - time: 210.8s - loss: 0.2729 - f1: 0.9227 - val_loss: 0.5078 - val_f1: 0.8398
Checkpoint saved: checkpoints/vit_epoch2.pt

Epoch 3/10 - time: 210.3s - loss: 0.0438 - f1: 0.9944 - val_loss: 0.4320 - val_f1: 0.8598
Checkpoint saved: checkpoints/vit_epoch3.pt

Epoch 4/10 - time: 210.1s - loss: 0.0075 - f1: 1.0000 - val_loss: 0.3996 - val_f1: 0.8727
Checkpoint saved: checkpoints/vit_epoch4.pt

Epoch 5/10 - time: 210.3s - loss: 0.0026 - f1: 1.0000 - val_loss: 0.4011 - val_f1: 0.8728
Checkpoint saved: checkpoints/vit_epoch5.pt

Epoch 6/10 - time: 209.9s - loss: 0.0015 - f1: 1.0000 - val_loss: 0.4072 - val_f1: 0.8716
Checkpoint saved: checkpoints/vit_epoch6.pt

Epoch 7/10 - time: 209.6s - loss: 0.0010 - f1: 1.0000 - val_loss: 0.4171 - val_f1: 0.8728
Checkpoint saved: checkpoints/vit_epoch7.pt

Epoch 8/10 - time: 209.6s - loss: 0.0007 - f1: 1.0000 -

In [25]:
from sklearn.metrics import classification_report

print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.89      0.93      0.91        87
      Boris_Kustodiev       0.76      0.83      0.80        66
     Camille_Pissarro       0.88      0.80      0.84        93
        Childe_Hassam       0.90      0.82      0.86        57
         Claude_Monet       0.86      0.93      0.89       140
          Edgar_Degas       0.78      0.88      0.82        64
        Eugene_Boudin       0.97      0.97      0.97        58
         Gustave_Dore       0.94      0.95      0.94        79
           Ilya_Repin       0.84      0.91      0.87        56
      Ivan_Aivazovsky       0.98      0.97      0.97        60
        Ivan_Shishkin       0.91      0.91      0.91        54
  John_Singer_Sargent       0.81      0.87      0.84        82
         Marc_Chagall       0.90      0.82      0.86        80
      Martiros_Saryan       0.81      0.90      0.85        60
     Nicholas_Roerich       0.91      0.95      0.93  

In [27]:
from sklearn.metrics import f1_score

model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Compute F1 score
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f"Test F1-score: {test_f1:.4f}")

Test F1-score: 0.8675


In [28]:
from sklearn.metrics import classification_report

print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.90      0.91      0.90        87
      Boris_Kustodiev       0.83      0.72      0.77        68
     Camille_Pissarro       0.90      0.83      0.86        94
        Childe_Hassam       0.92      0.76      0.83        59
         Claude_Monet       0.87      0.91      0.89       141
          Edgar_Degas       0.78      0.88      0.83        65
        Eugene_Boudin       0.90      0.93      0.92        59
         Gustave_Dore       0.95      0.95      0.95        80
           Ilya_Repin       0.77      0.83      0.80        58
      Ivan_Aivazovsky       0.97      0.90      0.93        62
        Ivan_Shishkin       0.87      0.93      0.90        56
  John_Singer_Sargent       0.84      0.96      0.90        83
         Marc_Chagall       0.96      0.89      0.92        81
      Martiros_Saryan       0.80      0.79      0.79        61
     Nicholas_Roerich       0.93      0.93      0.93  

# Version 2

In [23]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from sklearn.metrics import f1_score
import time
import torch
import os
from transformers import ViTForImageClassification, ViTConfig
import torch.nn as nn

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Labels ---
num_labels = len(train_dataset.classes)
id2label = {i: cls for i, cls in enumerate(train_dataset.classes)}
label2id = {cls: i for i, cls in enumerate(train_dataset.classes)}

# --- Model config with dropout ---
config = ViTConfig.from_pretrained('google/vit-base-patch16-224')
config.num_labels = num_labels
config.hidden_dropout_prob = 0.3          # feedforward dropout
config.attention_probs_dropout_prob = 0.2 # attention dropout
config.id2label = id2label
config.label2id = label2id

# --- Load pretrained model, ignore classifier mismatch ---
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True
)

# --- Extra dropout in classifier head ---
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(config.hidden_size, num_labels)
)

model.to(device)

# --- Optimizer & loss ---
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = CrossEntropyLoss()
epochs = 10

# --- Checkpoints directory ---
checkpoint_dir = "checkpoints/v2"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- Early stopping setup ---
best_val_f1 = 0
patience = 3
counter = 0

for epoch in range(epochs):
    start_time = time.time()

    # --- Training ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for pixel_values, labels in train_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()
            val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # --- Early stopping ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        counter = 0
        # Save the best model
        best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
        torch.save(model.state_dict(), best_model_path)
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # --- Epoch summary ---
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} - time: {epoch_time:.1f}s - "
          f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
          f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}")

    # --- Save epoch checkpoint ---
    checkpoint_path = os.path.join(checkpoint_dir, f"vit_epoch{epoch+1}.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}\n")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1/10 - time: 216.3s - loss: 1.6466 - f1: 0.4811 - val_loss: 1.0923 - val_f1: 0.6495
Checkpoint saved: checkpoints/v2/vit_epoch1.pt

Epoch 2/10 - time: 217.7s - loss: 0.8001 - f1: 0.7449 - val_loss: 0.7557 - val_f1: 0.7681
Checkpoint saved: checkpoints/v2/vit_epoch2.pt

Epoch 3/10 - time: 215.9s - loss: 0.5265 - f1: 0.8302 - val_loss: 0.8087 - val_f1: 0.7481
Checkpoint saved: checkpoints/v2/vit_epoch3.pt

Epoch 4/10 - time: 216.8s - loss: 0.3264 - f1: 0.8993 - val_loss: 0.6765 - val_f1: 0.7688
Checkpoint saved: checkpoints/v2/vit_epoch4.pt

Epoch 5/10 - time: 215.4s - loss: 0.2107 - f1: 0.9350 - val_loss: 0.7067 - val_f1: 0.7889
Checkpoint saved: checkpoints/v2/vit_epoch5.pt

Epoch 6/10 - time: 217.7s - loss: 0.1300 - f1: 0.9615 - val_loss: 0.6547 - val_f1: 0.8193
Checkpoint saved: checkpoints/v2/vit_epoch6.pt

Epoch 7/10 - time: 215.5s - loss: 0.0884 - f1: 0.9749 - val_loss: 0.6437 - val_f1: 0.8159
Checkpoint saved: checkpoints/v2/vit_epoch7.pt

Epoch 8/10 - time: 215.8s - loss: 

In [24]:
from sklearn.metrics import classification_report

print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.79      0.90      0.84        87
      Boris_Kustodiev       0.68      0.59      0.63        66
     Camille_Pissarro       0.77      0.78      0.78        93
        Childe_Hassam       0.94      0.51      0.66        57
         Claude_Monet       0.77      0.86      0.81       140
          Edgar_Degas       0.90      0.70      0.79        64
        Eugene_Boudin       0.93      0.90      0.91        58
         Gustave_Dore       0.96      0.92      0.94        79
           Ilya_Repin       0.85      0.62      0.72        56
      Ivan_Aivazovsky       0.94      1.00      0.97        60
        Ivan_Shishkin       0.86      0.89      0.87        54
  John_Singer_Sargent       0.79      0.83      0.81        82
         Marc_Chagall       0.95      0.66      0.78        80
      Martiros_Saryan       0.63      0.90      0.74        60
     Nicholas_Roerich       0.73      0.96      0.83  

In [25]:
from sklearn.metrics import f1_score

model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Compute F1 score
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f"Test F1-score: {test_f1:.4f}")

Test F1-score: 0.7992


In [26]:
from sklearn.metrics import classification_report

print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.83      0.86      0.85        87
      Boris_Kustodiev       0.80      0.65      0.72        68
     Camille_Pissarro       0.78      0.79      0.78        94
        Childe_Hassam       0.89      0.42      0.57        59
         Claude_Monet       0.77      0.87      0.82       141
          Edgar_Degas       0.78      0.69      0.73        65
        Eugene_Boudin       0.89      0.83      0.86        59
         Gustave_Dore       0.97      0.89      0.93        80
           Ilya_Repin       0.84      0.55      0.67        58
      Ivan_Aivazovsky       0.86      0.90      0.88        62
        Ivan_Shishkin       0.77      0.89      0.83        56
  John_Singer_Sargent       0.88      0.83      0.86        83
         Marc_Chagall       0.94      0.58      0.72        81
      Martiros_Saryan       0.64      0.90      0.75        61
     Nicholas_Roerich       0.74      0.96      0.84  

# Version 3

In [29]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import os

# --- Paths ---
base_path = "/teamspace/studios/this_studio/DeepLearning-NOVAIMS2026/data"
train_dir = os.path.join(base_path, "train")
val_dir   = os.path.join(base_path, "validation")
test_dir  = os.path.join(base_path, "test")

# --- Data Augmentation ---
# Start with one augmentation: RandomHorizontalFlip (common for WikiArt, see "Deep Ensemble Art Style Recognition")
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),  # horizontal flip augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# --- Load datasets ---
train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
val_dataset   = ImageFolder(root=val_dir, transform=val_transforms)
test_dataset  = ImageFolder(root=test_dir, transform=val_transforms)

# --- Data loaders ---
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# --- Labels info ---
num_labels = len(train_dataset.classes)
id2label = {i: cls for i, cls in enumerate(train_dataset.classes)}
label2id = {cls: i for i, cls in enumerate(train_dataset.classes)}

print(f"Datasets loaded successfully!")
print(f"Number of classes: {num_labels}")
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Datasets loaded successfully!
Number of classes: 23
Train batches: 583, Validation batches: 125, Test batches: 127


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import ViTForImageClassification, ViTConfig
from sklearn.metrics import f1_score
import time
import os

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model config ---
config = ViTConfig.from_pretrained('google/vit-base-patch16-224')
config.num_labels = num_labels
config.hidden_dropout_prob = 0.3          # feedforward dropout
config.attention_probs_dropout_prob = 0.2 # attention dropout
config.id2label = id2label
config.label2id = label2id

# --- Load pretrained ViT ---
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    config=config,
    ignore_mismatched_sizes=True  # classifier head reinit for 23 classes
)

# --- Extra dropout in classifier head ---
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(config.hidden_size, num_labels)
)

model.to(device)

# --- Optimizer & loss ---
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = CrossEntropyLoss()
epochs = 10

# --- Checkpoints directory ---
checkpoint_dir = "checkpoints/v3"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- Early stopping setup ---
best_val_f1 = 0
patience = 4
counter = 0

for epoch in range(epochs):
    start_time = time.time()

    # --- Training ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for pixel_values, labels in train_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()
            val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # --- Early stopping ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        counter = 0
        best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
        torch.save(model.state_dict(), best_model_path)
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # --- Epoch summary ---
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} - time: {epoch_time:.1f}s - "
          f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
          f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}")

    # --- Save epoch checkpoint ---
    checkpoint_path = os.path.join(checkpoint_dir, f"vit_epoch{epoch+1}.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}\n")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch 1/10 - time: 157.9s - loss: 1.6562 - f1: 0.4802 - val_loss: 1.1180 - val_f1: 0.6302
Checkpoint saved: checkpoints/version2/vit_epoch1.pt

Epoch 2/10 - time: 159.3s - loss: 0.7956 - f1: 0.7508 - val_loss: 0.9054 - val_f1: 0.7111
Checkpoint saved: checkpoints/version2/vit_epoch2.pt

Epoch 3/10 - time: 158.9s - loss: 0.5345 - f1: 0.8311 - val_loss: 0.6844 - val_f1: 0.7723
Checkpoint saved: checkpoints/version2/vit_epoch3.pt

Epoch 4/10 - time: 158.9s - loss: 0.3372 - f1: 0.8981 - val_loss: 0.6836 - val_f1: 0.7852
Checkpoint saved: checkpoints/version2/vit_epoch4.pt

Epoch 5/10 - time: 159.1s - loss: 0.2103 - f1: 0.9374 - val_loss: 0.6783 - val_f1: 0.7928
Checkpoint saved: checkpoints/version2/vit_epoch5.pt

Epoch 6/10 - time: 158.1s - loss: 0.1422 - f1: 0.9576 - val_loss: 0.7627 - val_f1: 0.7786
Checkpoint saved: checkpoints/version2/vit_epoch6.pt

Epoch 7/10 - time: 158.7s - loss: 0.0928 - f1: 0.9747 - val_loss: 0.7202 - val_f1: 0.7937
Checkpoint saved: checkpoints/version2/vit_epo

In [31]:
from sklearn.metrics import classification_report

print(classification_report(
    val_labels,
    val_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.84      0.91      0.87        87
      Boris_Kustodiev       0.52      0.85      0.65        66
     Camille_Pissarro       0.84      0.67      0.74        93
        Childe_Hassam       0.89      0.68      0.77        57
         Claude_Monet       0.76      0.91      0.82       140
          Edgar_Degas       0.75      0.83      0.79        64
        Eugene_Boudin       0.96      0.91      0.94        58
         Gustave_Dore       0.96      0.91      0.94        79
           Ilya_Repin       0.74      0.71      0.73        56
      Ivan_Aivazovsky       0.97      1.00      0.98        60
        Ivan_Shishkin       0.75      0.93      0.83        54
  John_Singer_Sargent       0.95      0.63      0.76        82
         Marc_Chagall       0.93      0.71      0.81        80
      Martiros_Saryan       0.74      0.88      0.80        60
     Nicholas_Roerich       0.72      0.94      0.82  

In [32]:
from sklearn.metrics import f1_score

model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Compute F1 score
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f"Test F1-score: {test_f1:.4f}")

Test F1-score: 0.7735


In [37]:
from sklearn.metrics import classification_report

print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.79      0.90      0.84        87
      Boris_Kustodiev       0.57      0.84      0.68        68
     Camille_Pissarro       0.85      0.78      0.81        94
        Childe_Hassam       0.87      0.58      0.69        59
         Claude_Monet       0.74      0.85      0.79       141
          Edgar_Degas       0.65      0.69      0.67        65
        Eugene_Boudin       0.92      0.75      0.82        59
         Gustave_Dore       0.96      0.91      0.94        80
           Ilya_Repin       0.73      0.64      0.68        58
      Ivan_Aivazovsky       0.92      0.90      0.91        62
        Ivan_Shishkin       0.64      0.91      0.75        56
  John_Singer_Sargent       0.95      0.67      0.79        83
         Marc_Chagall       0.85      0.79      0.82        81
      Martiros_Saryan       0.73      0.77      0.75        61
     Nicholas_Roerich       0.78      0.94      0.85  

# Things to check if cuda is not working

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [12]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[]

In [11]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPUs: []


In [10]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


In [9]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
1
0
NVIDIA L4


In [13]:
print(torch.version.cuda)

13.0


In [8]:
print(torch.cuda.is_available())

True
